In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here are several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs PAK.mp4
/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4
/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs SA.mp4
/kaggle/input/datasets/loak2055/yolo-weights/best.pt
/kaggle/input/datasets/kuldipkmanvar/transcript-indvseng/transcript_with_timestamps.txt


# Installations

In [2]:
# Whisper not needed — using pre-existing transcript
# !pip install -q faster-whisper

# Audio Loudness based Highlight Generation

In [3]:
import os
import json
import math
import subprocess
from typing import Optional
from pathlib import Path
import numpy as np
import soundfile as sf
from scipy.signal import find_peaks
from scipy.ndimage import gaussian_filter1d, median_filter


# =========================================================
# Config
# =========================================================

DEFAULT_SR = 16000
WINDOW_SEC = 1.25      # analysis window
HOP_SEC = 0.25         # analysis hop
MERGE_GAP_SEC = 8.0
MIN_EVENT_SEC = 2.0

# Peak detection tuning
PEAK_PROMINENCE = 0.8
PEAK_DISTANCE_SEC = 6.0

# Sustained loudness tuning
SUSTAIN_Z_THRESHOLD = 1.0
SUSTAIN_MIN_SEC = 2.0

# Clip padding around detected highlight moments
PRE_PAD_SEC = 15.0
POST_PAD_SEC = 5.0


# =========================================================
# Audio extraction
# =========================================================
def extract_audio_ffmpeg(video_path: str, wav_path: str, sr: int = DEFAULT_SR):
    """
    Extract mono WAV audio from a video using ffmpeg.
    """
    cmd = [
        "ffmpeg",
        "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", str(sr),
        "-acodec", "pcm_s16le",
        wav_path
    ]
    subprocess.run(cmd, check=True)


# =========================================================
# Time formatting
# =========================================================

def seconds_to_hms(seconds: float) -> str:
    seconds = int(round(float(seconds)))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    return f"{h:02d}:{m:02d}:{s:02d}"

# =========================================================
# Feature helpers
# =========================================================

def robust_zscore(x: np.ndarray) -> np.ndarray:
    """
    Robust z-score using median and MAD.
    """
    x = np.asarray(x, dtype=np.float32)
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    if mad < 1e-8:
        return np.zeros_like(x)
    return 0.6745 * (x - med) / mad

def contiguous_true_runs(mask: np.ndarray):
    """
    Return list of (start_idx, end_idx) inclusive runs where mask is True.
    """
    runs = []
    if len(mask) == 0:
        return runs

    start = None
    for i, val in enumerate(mask):
        if val and start is None:
            start = i
        elif not val and start is not None:
            runs.append((start, i - 1))
            start = None

    if start is not None:
        runs.append((start, len(mask) - 1))

    return runs


def read_window(sf_handle, start_frame: int, n_frames: int):
    """
    Read a fixed-size mono window from an opened SoundFile handle.
    Returns None if the window is too short.
    """
    sf_handle.seek(start_frame)
    x = sf_handle.read(n_frames, dtype="float32", always_2d=True)
    if x.shape[0] < n_frames:
        return None
    return x.mean(axis=1)  # mono


def compute_features_streaming(
    wav_path: str,
    win_sec: float = WINDOW_SEC,
    hop_sec: float = HOP_SEC,
):
    """
    Stream through the WAV file and compute frame-level features.
    Returns a dict with time axis and feature arrays.
    """
    times = []
    rms_db = []
    peak_db = []
    crest_db = []
    flux = []
    centroid_norm = []

    with sf.SoundFile(wav_path) as f:
        sr = f.samplerate
        win_frames = int(win_sec * sr)
        hop_frames = int(hop_sec * sr)

        total_frames = len(f)
        starts = range(0, max(1, total_frames - win_frames + 1), hop_frames)

        prev_mag = None

        for start in starts:
            x = read_window(f, start, win_frames)
            if x is None:
                break

            t = start / sr
            times.append(t)

            # Loudness
            rms = float(np.sqrt(np.mean(x * x) + 1e-12))
            peak = float(np.max(np.abs(x)) + 1e-12)
            rdb = 20.0 * math.log10(rms + 1e-12)
            pdb = 20.0 * math.log10(peak + 1e-12)
            cdb = pdb - rdb

            rms_db.append(rdb)
            peak_db.append(pdb)
            crest_db.append(cdb)

            # Spectral flux and centroid
            window = np.hanning(len(x)).astype(np.float32)
            spec = np.abs(np.fft.rfft(x * window)).astype(np.float32)
            spec_sum = float(spec.sum() + 1e-12)
            mag = spec / spec_sum

            if prev_mag is None:
                flx = 0.0
            else:
                flx = float(np.sqrt(np.mean(np.square(np.maximum(mag - prev_mag, 0.0)))))
            prev_mag = mag
            flux.append(flx)

            freqs = np.fft.rfftfreq(len(x), d=1.0 / sr)
            cent = float((freqs * mag).sum() / (mag.sum() + 1e-12))
            centroid_norm.append(cent / (sr / 2.0))

    times = np.asarray(times, dtype=np.float32)
    rms_db = np.asarray(rms_db, dtype=np.float32)
    peak_db = np.asarray(peak_db, dtype=np.float32)
    crest_db = np.asarray(crest_db, dtype=np.float32)
    flux = np.asarray(flux, dtype=np.float32)
    centroid_norm = np.asarray(centroid_norm, dtype=np.float32)

    delta_db = np.concatenate([[0.0], np.diff(rms_db)]).astype(np.float32)

    return {
        "sr": sr,
        "times": times,
        "rms_db": rms_db,
        "peak_db": peak_db,
        "crest_db": crest_db,
        "flux": flux,
        "centroid_norm": centroid_norm,
        "delta_db": delta_db,
        "win_sec": win_sec,
        "hop_sec": hop_sec,
    }


# =========================================================
# Detection
# =========================================================

def build_composite_score(features: dict):
    """
    Combine several audio cues into one robust score.
    """
    z_rms = robust_zscore(features["rms_db"])
    z_flux = robust_zscore(features["flux"])
    z_delta = robust_zscore(features["delta_db"])
    z_cent = robust_zscore(features["centroid_norm"])
    z_crest = robust_zscore(features["crest_db"])

    score = (
        0.55 * z_rms +
        0.25 * z_flux +
        0.10 * z_delta +
        0.05 * z_cent +
        0.05 * z_crest
    ).astype(np.float32)

    score = median_filter(score, size=5)
    score = gaussian_filter1d(score, sigma=1.0)

    return score


def detect_peak_candidates(features: dict, score: np.ndarray):
    """
    Detect local maxima in the composite score.
    """
    hop_sec = features["hop_sec"]
    times = features["times"]

    min_distance_frames = max(1, int(PEAK_DISTANCE_SEC / hop_sec))
    peaks, props = find_peaks(
        score,
        prominence=PEAK_PROMINENCE,
        distance=min_distance_frames,
    )

    candidates = []
    for idx in peaks:
        t = float(times[idx])
        candidates.append({
            "start": max(0.0, t - PRE_PAD_SEC),
            "end": t + POST_PAD_SEC,
            "peak_time": t,
            "score": float(score[idx]),
            "source": ["audio_peak"],
        })
    return candidates


def detect_sustained_loud_regions(features: dict, score: np.ndarray):
    """
    Detect longer noisy/energetic spans: crowd roar, celebration, extended commentary surge.
    Uses both absolute loudness and the robust composite score.
    """
    hop_sec = features["hop_sec"]
    times = features["times"]

    z_score = robust_zscore(score)
    rms_db = features["rms_db"]

    abs_floor = np.percentile(rms_db, 80)
    rel_mask = z_score >= SUSTAIN_Z_THRESHOLD
    abs_mask = rms_db >= abs_floor

    mask = rel_mask & abs_mask
    runs = contiguous_true_runs(mask)

    candidates = []

    for s, e in runs:
        duration = (e - s + 1) * hop_sec
        if duration < SUSTAIN_MIN_SEC:
            continue

        local_idx = s + int(np.argmax(score[s:e + 1]))
        peak_t = float(times[local_idx])

        candidates.append({
            "start": max(0.0, float(times[s]) - 4.0),
            "end": float(times[e]) + 6.0,
            "peak_time": peak_t,
            "score": float(np.max(score[s:e + 1])),
            "source": ["audio_sustained"],
        })

    return candidates


def load_transcript_candidates(transcript_json_path: str):
    """
    Optional fusion with transcript-based highlight candidates.
    Accepts either:
    - {"highlights":[...]}
    - [...]
    """
    if not transcript_json_path:
        return []

    data = json.loads(Path(transcript_json_path).read_text(encoding="utf-8"))

    if isinstance(data, dict) and "highlights" in data:
        items = data["highlights"]
    elif isinstance(data, list):
        items = data
    else:
        return []

    candidates = []
    for item in items:
        start = float(item.get("start", item.get("start_time", 0.0)))
        end = float(item.get("end", item.get("end_time", start + 10.0)))
        conf = float(item.get("confidence", 0.7))

        candidates.append({
            "start": max(0.0, start - 4.0),
            "end": end + 6.0,
            "peak_time": start,
            "score": 2.0 + conf,
            "source": ["transcript"],
        })

    return candidates


# =========================================================
# Merging
# =========================================================

def merge_windows(windows, gap_sec: float = MERGE_GAP_SEC):
    """
    Merge overlapping or near-overlapping windows.
    Returns windows in numeric seconds for internal processing.
    """
    if not windows:
        return []

    windows = sorted(windows, key=lambda x: (x["start"], x["end"]))
    merged = [windows[0]]

    for cur in windows[1:]:
        prev = merged[-1]

        if cur["start"] <= prev["end"] + gap_sec:
            prev["start"] = round(min(prev["start"], cur["start"]), 2)
            prev["end"] = round(max(prev["end"], cur["end"]), 2)
            prev["peak_time"] = round(min(prev["peak_time"], cur["peak_time"]), 2)
            prev["score"] = round(max(prev["score"], cur["score"]), 4)
            prev["source"] = sorted(set(prev["source"]) | set(cur["source"]))
        else:
            merged.append(cur)

    cleaned = []
    for w in merged:
        duration = w["end"] - w["start"]
        if duration >= MIN_EVENT_SEC:
            cleaned.append({
                "start": round(w["start"], 2),
                "end": round(w["end"], 2),
                "peak_time": round(w["peak_time"], 2),
                "score": round(w["score"], 4),
                "source": w["source"],
            })
    return cleaned


def format_windows_for_output(windows):
    """
    Convert numeric seconds into HH:MM:SS strings for final JSON output.
    Keeps raw seconds too.
    """
    formatted = []
    for w in windows:
        start_sec = float(w["start"])
        end_sec = float(w["end"])
        peak_sec = float(w["peak_time"])

        formatted.append({
            "start": seconds_to_hms(start_sec),
            "end": seconds_to_hms(end_sec),
            "peak_time": seconds_to_hms(peak_sec),
            "start_sec": round(start_sec, 2),
            "end_sec": round(end_sec, 2),
            "peak_time_sec": round(peak_sec, 2),
            "score": round(float(w["score"]), 4),
            "source": w["source"],
        })
    return formatted


# =========================================================
# Main pipeline
# =========================================================

def analyze_audio_for_highlights(
    video_path: str,
    output_json: str = "audio_highlights.json",
    work_dir: str = "./audio_work",
    transcript_json_path: Optional[str] = None,
):
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)

    wav_path = str(work_dir / "audio.wav")
    print("Extracting audio...")
    extract_audio_ffmpeg(video_path, wav_path, sr=DEFAULT_SR)

    print("Computing streaming audio features...")
    features = compute_features_streaming(wav_path, win_sec=WINDOW_SEC, hop_sec=HOP_SEC)

    print("Building composite score...")
    score = build_composite_score(features)

    print("Detecting peak-based candidates...")
    peak_candidates = detect_peak_candidates(features, score)

    print("Detecting sustained loudness candidates...")
    sustain_candidates = detect_sustained_loud_regions(features, score)

    transcript_candidates = []
    if transcript_json_path:
        print("Loading transcript candidates...")
        transcript_candidates = load_transcript_candidates(transcript_json_path)

    all_candidates = peak_candidates + sustain_candidates + transcript_candidates

    merged = merge_windows(all_candidates, gap_sec=MERGE_GAP_SEC)

    merged = sorted(merged, key=lambda x: (-x["score"], x["start"]))

    formatted_highlights = format_windows_for_output(merged)

    payload = {
        "video_path": video_path,
        "sample_rate": DEFAULT_SR,
        "window_sec": WINDOW_SEC,
        "hop_sec": HOP_SEC,
        "num_raw_candidates": len(all_candidates),
        "num_final_highlights": len(formatted_highlights),
        "highlights": formatted_highlights,
    }

    Path(output_json).write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(f"Saved {len(formatted_highlights)} highlight windows to {output_json}")
    return payload


In [4]:
# (Audio analysis is invoked from the integration cell below)

# Transcript Generation

In [5]:
# ============================================================
# SKIPPED: Whisper transcription
# Using pre-uploaded transcript from Kaggle dataset instead.
# This saves ~60 minutes of GPU time.
# ============================================================
print("Whisper transcription SKIPPED — using pre-existing transcript file.")

Whisper transcription SKIPPED — using pre-existing transcript file.


In [6]:
# Transcript already exists at:
# /kaggle/input/datasets/yashashishpandit/transcript-ee655/transcript_with_timestamps.txt
# (or wherever your transcript dataset is mounted)
print("Transcript file ready — no transcription needed.")

Transcript file ready — no transcription needed.


In [7]:
# No Whisper model was loaded, so nothing to free.
# GPU memory is fully available for YOLO + EasyOCR.
import torch
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total_gb = props.total_memory / 1e9
    print(f"GPU: {props.name} — {total_gb:.1f} GB total VRAM available for YOLO + EasyOCR.")
else:
    print("No CUDA GPU detected — will run on CPU.")


# LLM Based Transcript Highlight Extraction

In [ ]:
!pip install google-genai

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["GEMINI_API_KEY"] = UserSecretsClient().get_secret("GEMINI_API_KEY")

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Literal, Optional

from google import genai
from pydantic import BaseModel, Field, ValidationError


# -----------------------------
# Configuration
# -----------------------------

CHUNK_MODEL = "gemini-2.5-flash"
FINAL_MODEL = "gemini-2.5-pro"

WINDOW_SECONDS = 90.0
OVERLAP_SECONDS = 18.0

# For merging near-duplicate LLM outputs before the final pass
MERGE_GAP_SECONDS = 10.0


EventType = Literal["wicket", "six", "four", "review", "milestone", "other"]


# -----------------------------
# Data models
# -----------------------------

class TranscriptWord(BaseModel):
    start: float
    end: float
    word: str


class TranscriptSegment(BaseModel):
    start: float
    end: float
    text: str
    words: List[TranscriptWord] = []


class HighlightCandidate(BaseModel):
    event_type: EventType
    start: float = Field(description="Absolute match timestamp in seconds.")
    end: float = Field(description="Absolute match timestamp in seconds.")
    confidence: float = Field(ge=0.0, le=1.0)
    evidence: str
    reason: Optional[str] = None


# -----------------------------
# Parsing the transcript file
# -----------------------------

SEGMENT_RE = re.compile(r"^\[(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\]\s+(.*)$")
WORD_RE = re.compile(r"^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*:\s*(.*)$")


def parse_transcript_file(path: str) -> List[TranscriptSegment]:
    """
    Parses files like:
      [0.21 - 2.43]  First ball being bowled to Jason Roy.
          0.21 - 0.71:  First
          0.71 - 0.99:  ball
    """
    lines = Path(path).read_text(encoding="utf-8", errors="ignore").splitlines()

    segments: List[TranscriptSegment] = []
    current: Optional[TranscriptSegment] = None

    for line in lines:
        seg_m = SEGMENT_RE.match(line)
        if seg_m:
            if current is not None:
                segments.append(current)
            current = TranscriptSegment(
                start=float(seg_m.group(1)),
                end=float(seg_m.group(2)),
                text=seg_m.group(3).strip(),
                words=[],
            )
            continue

        word_m = WORD_RE.match(line)
        if word_m and current is not None:
            current.words.append(
                TranscriptWord(
                    start=float(word_m.group(1)),
                    end=float(word_m.group(2)),
                    word=word_m.group(3).strip(),
                )
            )

    if current is not None:
        segments.append(current)

    return segments


# -----------------------------
# Chunking
# -----------------------------

def build_overlapping_chunks(
    segments: List[TranscriptSegment],
    window_seconds: float = WINDOW_SECONDS,
    overlap_seconds: float = OVERLAP_SECONDS,
):
    if not segments:
        return

    segments = sorted(segments, key=lambda s: (s.start, s.end))
    t0 = segments[0].start
    t_end = segments[-1].end
    step = max(1.0, window_seconds - overlap_seconds)
    
    # `i` is a non-resetting forward scan pointer. It intentionally never resets
    # between windows because segments behind the current window start are never
    # needed again. Do NOT call this generator twice on the same segment list
    # without re-instantiating it, as `i` will not rewind.

    i = 0
    n = len(segments)
    chunk_id = 0

    while t0 < t_end:
        # Advance pointer past segments that end before this window
        while i < n and segments[i].end <= t0:
            i += 1

        j = i
        chunk_segments: List[TranscriptSegment] = []
        window_end = t0 + window_seconds

        while j < n and segments[j].start < window_end:
            if segments[j].end > t0:
                chunk_segments.append(segments[j])
            j += 1

        if chunk_segments:
            yield {
                "chunk_id": chunk_id,
                "chunk_start": t0,
                "chunk_end": window_end,
                "segments": chunk_segments,
            }
            chunk_id += 1

        t0 += step


def render_chunk(chunk) -> str:
    lines = []
    lines.append(
        f"CHUNK {chunk['chunk_id']} | WINDOW [{chunk['chunk_start']:.2f} - {chunk['chunk_end']:.2f}] seconds"
    )
    lines.append(
        "Task: Identify only real cricket highlight moments in this chunk. "
        "Prefer wickets, sixes, fours, reviews, milestones, and clearly decisive moments."
    )
    lines.append(
        "Return absolute timestamps from the match transcript, not relative timestamps."
    )
    lines.append("Transcript:")
    for seg in chunk["segments"]:
        lines.append(f"[{seg.start:.2f} - {seg.end:.2f}] {seg.text}")
    return "\n".join(lines)


# -----------------------------
# Gemini calls
# -----------------------------

def get_client() -> genai.Client:
    # The SDK will read GEMINI_API_KEY from the environment automatically.
    api_key = os.getenv("GEMINI_API_KEY")
    if api_key:
        return genai.Client(api_key=api_key)
    return genai.Client()


def parse_structured_response(text: str, model_cls):
    """
    Structured output usually returns valid JSON, but this makes the pipeline
    more resilient if the model wraps the response in extra text.
    """
    text = text.strip()
    try:
        return model_cls.model_validate_json(text)
    except ValidationError:
        pass

    # Try to recover JSON from fenced or noisy output.
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start : end + 1]
        return model_cls.model_validate_json(candidate)

    raise ValueError(f"Could not parse structured JSON response:\n{text[:1000]}")


def extract_chunk_highlights(client: genai.Client, chunk_text: str) -> List[HighlightCandidate]:
    class ChunkResult(BaseModel):
        highlights: List[HighlightCandidate] = []

    prompt = f"""
You are analyzing a cricket commentary transcript chunk.

Return JSON only that matches the schema.
Keep the output strictly focused on highlight moments:
- wicket
- six
- four
- review
- milestone
- other decisive moments

Rules:
- Use the transcript's absolute timestamps exactly.
- According to the highlight moment and the commentry adjust the start and end times. 
- Analyse how the commentry is correlated to the actual moment on ground so that the highlight moment is accurately captured between the start and end times.
- Do not invent events.
- Ignore pitch discussion, player introductions, generic analysis, and filler.
- Return up to 6 candidates for this chunk.
- If nothing important happens in the chunk, return an empty list.

Transcript chunk:
{chunk_text}
""".strip()

    response = client.models.generate_content(
        model=CHUNK_MODEL,
        contents=prompt,
        config={
            "temperature": 0.2,
            "response_mime_type": "application/json",
            "response_json_schema": ChunkResult.model_json_schema(),
        },
    )

    parsed = parse_structured_response(response.text, ChunkResult)
    return parsed.highlights


def merge_close_candidates(candidates: List[HighlightCandidate]) -> List[HighlightCandidate]:
    if not candidates:
        return []

    priority = {"wicket": 5, "six": 4, "four": 3, "review": 2, "milestone": 2, "other": 1}
    candidates = sorted(candidates, key=lambda x: (x.start, x.end))

    merged: List[HighlightCandidate] = [candidates[0]]

    for cur in candidates[1:]:
        prev = merged[-1]

        overlap = min(prev.end, cur.end) - max(prev.start, cur.start)
        gap = cur.start - prev.end

        should_merge = overlap >= 0 or gap <= MERGE_GAP_SECONDS

        if should_merge:
            # Merge into a single wider window
            prev.start = round(min(prev.start, cur.start), 2)
            prev.end = round(max(prev.end, cur.end), 2)
            prev.confidence = round(max(prev.confidence, cur.confidence), 3)

            # Keep the more important label if they differ
            if priority[cur.event_type] > priority[prev.event_type]:
                prev.event_type = cur.event_type

            # Combine evidence compactly
            prev.evidence = f"{prev.evidence} | {cur.evidence}".strip(" |")
            if cur.reason:
                prev.reason = f"{prev.reason} | {cur.reason}" if prev.reason else cur.reason
        else:
            merged.append(cur)

    return merged


def final_select_highlights(client: genai.Client, merged_candidates: List[HighlightCandidate]) -> List[HighlightCandidate]:
    class FinalResult(BaseModel):
        highlights: List[HighlightCandidate]

    candidate_payload = [
        {
            "event_type": c.event_type,
            "start": c.start,
            "end": c.end,
            "confidence": c.confidence,
            "evidence": c.evidence,
            "reason": c.reason,
        }
        for c in merged_candidates
    ]

    prompt = f"""
You are given candidate cricket highlight windows extracted from an entire match transcript.

Your job:
- Remove duplicates and near-duplicates.
- Keep only the best highlight windows.
- Improve the final selection quality.
- Prioritize wickets, sixes, fours, reviews, milestones, and other truly important moments.
- Ignore commentary-only passages, analysis, and duplicates.
- Sort the final output by match time.

Return JSON only that matches the schema.

Candidate windows:
{json.dumps(candidate_payload, indent=2)}
""".strip()

    response = client.models.generate_content(
        model=FINAL_MODEL,
        contents=prompt,
        config={
            "temperature": 0.1,
            "response_mime_type": "application/json",
            "response_json_schema": FinalResult.model_json_schema(),
        },
    )

    parsed = parse_structured_response(response.text, FinalResult)
    return parsed.highlights


# -----------------------------
# Main pipeline
# -----------------------------

def run_pipeline(input_path: str, output_path: str):
    client = get_client()
    segments = parse_transcript_file(input_path)

    all_candidates: List[HighlightCandidate] = []

    for chunk in build_overlapping_chunks(segments):
        chunk_text = render_chunk(chunk)
        try:
            chunk_candidates = extract_chunk_highlights(client, chunk_text)
            all_candidates.extend(chunk_candidates)
            print(f"Chunk {chunk['chunk_id']}: {len(chunk_candidates)} candidates")
        except Exception as e:
            print(f"Chunk {chunk['chunk_id']} failed: {e}")

    # First merge to reduce duplicates before the final reasoning pass
    merged_candidates = merge_close_candidates(sorted(all_candidates, key=lambda x: (x.start, x.end)))

    # Final model pass for ranking, deduping, and boundary cleanup
    final_highlights = final_select_highlights(client, merged_candidates)

    # Sort final output by time
    final_highlights = sorted(final_highlights, key=lambda x: (x.start, x.end))

    payload = {
        "source_file": input_path,
        "chunk_model": CHUNK_MODEL,
        "final_model": FINAL_MODEL,
        "num_segments": len(segments),
        "num_candidates": len(all_candidates),
        "num_final_highlights": len(final_highlights),
        "highlights": [h.model_dump() for h in final_highlights],
    }

    Path(output_path).write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"\nSaved final highlights to: {output_path}")
    print(f"Final highlight count: {len(final_highlights)}")




In [ ]:
# (Gemini transcript pipeline is invoked from the integration cell below)

# YOLO + OCR

In [ ]:
!pip install -q ultralytics easyocr opencv-python-headless tqdm

import os
import cv2
import re
import torch
import numpy as np
from datetime import timedelta
from ultralytics import YOLO
import easyocr
from tqdm.notebook import tqdm

WEIGHTS_PATH = '/kaggle/input/datasets/loak2055/yolo-weights/best.pt'
VIDEO_PATH = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'

device_yolo = "cuda:0" if torch.cuda.device_count() > 0 else "cpu"
device_ocr = "cuda:1" if torch.cuda.device_count() > 1 else device_yolo

print(f"YOLO running on: {device_yolo}")
print(f"EasyOCR running on: {device_ocr}")


In [ ]:
trained_model = YOLO(WEIGHTS_PATH)
trained_model.to(device_yolo)

reader = easyocr.Reader(['en'], gpu=True)
print("Models loaded and ready for inference.")

In [ ]:
class SceneChangeDetector:
    """Lightweight histogram-based scene-change detector.
    
    Computes an HSV colour histogram per (downscaled) frame and flags
    timestamps where consecutive histograms differ by more than `threshold`
    (chi-square distance on the normalised histogram vector).
    """

    def __init__(self, video_path, threshold=0.5, resize=(320, 240), merge_gap=0.2):
        self.video_path = video_path
        self.threshold = threshold
        self.resize = resize
        self.merge_gap = merge_gap
        self.scene_times = []

    # ---- core detection ----
    @staticmethod
    def _frame_hist(frame):
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [50, 60], [0, 180, 0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        return hist

    def detect(self):
        """Run scene-change detection over the full video. Returns sorted list of timestamps (seconds)."""
        cap = cv2.VideoCapture(self.video_path)
        if not cap.isOpened():
            print("⚠️  Cannot open video for scene detection")
            return []
        fps = cap.get(cv2.CAP_PROP_FPS)
        prev_hist = None
        frame_idx = 0
        raw_times = []

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        pbar = tqdm(total=total_frames, desc="Scene detection", leave=False, mininterval=2.0)

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            small = cv2.resize(frame, self.resize)
            hist = self._frame_hist(small)
            if prev_hist is not None:
                diff = np.linalg.norm(hist - prev_hist)
                if diff > self.threshold:
                    raw_times.append(frame_idx / fps)
            prev_hist = hist
            frame_idx += 1
            pbar.update(1)

        cap.release()
        pbar.close()

        # merge timestamps closer than merge_gap seconds
        self.scene_times = self._merge_close(raw_times)
        print(f"Scene detection: {len(self.scene_times)} boundaries found")
        return self.scene_times

    def _merge_close(self, times):
        if not times:
            return []
        merged = []
        cluster = [times[0]]
        for t in times[1:]:
            if t - cluster[-1] <= self.merge_gap:
                cluster.append(t)
            else:
                merged.append(sum(cluster) / len(cluster))
                cluster = [t]
        if cluster:
            merged.append(sum(cluster) / len(cluster))
        return merged

    # ---- boundary snapping helpers ----
    def snap_start(self, desired_start, min_stable=3.0):
        """Return the nearest scene-change >= desired_start whose NEXT scene-change
        is at least min_stable seconds later (i.e. the clip starts on a stable shot)."""
        for i, s in enumerate(self.scene_times):
            if s >= desired_start:
                if i < len(self.scene_times) - 1 and (self.scene_times[i + 1] - s) >= min_stable:
                    return s
                break
        return desired_start  # fallback: keep original

    def snap_end(self, desired_end, min_stable=3.0):
        """Return the nearest scene-change <= desired_end whose PREVIOUS scene-change
        is at least min_stable seconds earlier (i.e. the clip ends on a stable shot)."""
        for i in range(len(self.scene_times) - 1, -1, -1):
            s = self.scene_times[i]
            if s <= desired_end:
                if i > 0 and (s - self.scene_times[i - 1]) >= min_stable:
                    return s
                break
        return desired_end  # fallback: keep original


class CricketHighlightPipeline:
    def __init__(self, yolo_model, ocr_reader, video_path, max_duration_mins=None, sample_fps=1, persistence=3):
        self.yolo_model = yolo_model
        self.reader = ocr_reader
        self.video_path = video_path
        self.sample_fps = sample_fps
        self.persistence_limit = persistence
        
        if max_duration_mins is not None:
            self.max_duration_sec = max_duration_mins * 60
        else:
            self.max_duration_sec = float('inf')
            
        self.last_valid_score = {"runs": None, "wickets": None}
        self.highlights = []
        self.score_buffer = []
        self.innings_active = False

    # ---- NEW: enhanced OCR preprocessing ----
    @staticmethod
    def preprocess_roi(roi):
        """Upscale small crops + Otsu binarisation for cleaner OCR input."""
        h, w = roi.shape[:2]
        min_side = min(h, w)
        if min_side < 500:
            scale = 500 / min_side
            roi = cv2.resize(roi, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_LINEAR)
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        return thresh

    def preprocess_ocr_text(self, text):
        replacements = {'O': '0', 'o': '0', 'l': '1', 'I': '1', 'i': '1', 'S': '5', 's': '5', 'B': '8', 'Z': '2', 'z': '2'}
        for k, v in replacements.items():
            text = text.replace(k, v)
        return re.sub(r'[^\d/-]', '', text)

    def extract_score(self, text):
        cleaned_text = self.preprocess_ocr_text(text)
        match = re.search(r'(\d{1,3})\s*[/|-]\s*(\d{1,2})', cleaned_text)
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    def get_stable_score(self, runs, wickets):
        self.score_buffer.append((runs, wickets))
        if len(self.score_buffer) > self.persistence_limit:
            self.score_buffer.pop(0)
        
        if len(self.score_buffer) == self.persistence_limit and len(set(self.score_buffer)) == 1:
            return self.score_buffer[0]
        return None, None

    def validate_and_update(self, current_runs, current_wickets):
        if current_runs == 0 and current_wickets == 0:
            is_prev_valid = self.last_valid_score["runs"] is not None and self.last_valid_score["runs"] > 0
            is_initial = self.last_valid_score["runs"] is None
            
            if (is_initial or is_prev_valid) and not self.innings_active:
                self.last_valid_score = {"runs": 0, "wickets": 0}
                self.innings_active = True
                return True, "New Innings Detected"
            elif self.innings_active:
                return False, None

        if current_runs > 0 or current_wickets > 0:
            self.innings_active = False

        if self.last_valid_score["runs"] is None:
            self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            return False, None

        prev_runs = self.last_valid_score["runs"]
        prev_wickets = self.last_valid_score["wickets"]
        
        run_diff = current_runs - prev_runs
        wicket_diff = current_wickets - prev_wickets

        if 0 <= run_diff <= 7 and 0 <= wicket_diff <= 1:
            event_type = None
            if run_diff == 4: event_type = "Four"
            elif run_diff == 6: event_type = "Six"
            elif wicket_diff == 1: event_type = "Wicket"
            
            if run_diff > 0 or wicket_diff > 0:
                self.last_valid_score = {"runs": current_runs, "wickets": current_wickets}
            
            return (event_type is not None), event_type
        
        return False, None

    def process_video(self):
        cap = cv2.VideoCapture(self.video_path)
        fps = cap.get(cv2.CAP_PROP_FPS)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        
        limit_frames = total_frames if self.max_duration_sec == float('inf') else int(self.max_duration_sec * fps)
        total_to_process = min(total_frames, limit_frames)
        
        frame_interval = max(1, int(fps / self.sample_fps))
        
        pbar = tqdm(total=total_to_process, desc="Analyzing Match", leave=True, mininterval=1.0)
        
        frame_count = 0
        while cap.isOpened():
            if frame_count % frame_interval == 0:
                ret, frame = cap.read()
            else:
                ret = cap.grab()
                frame_count += 1
                if frame_count % 10 == 0:
                    pbar.update(10)
                continue

            current_time_sec = frame_count / fps
            if not ret or current_time_sec > self.max_duration_sec:
                break
                
            h, w = frame.shape[:2]
            lower_third = frame[int(h * 0.75):h, 0:w]
            results = self.yolo_model.predict(lower_third, conf=0.45, verbose=False, device=0)
            
            if results and len(results[0].boxes) > 0:
                boxes = sorted(results[0].boxes, key=lambda x: x.conf[0].item(), reverse=True)
                x1, y1, x2, y2 = map(int, boxes[0].xyxy[0])
                roi = lower_third[y1:y2, x1:x2]
                
                if roi.size > 0:
                    # ---- CHANGED: use enhanced preprocessing instead of plain greyscale ----
                    processed_roi = self.preprocess_roi(roi)
                    ocr_res = self.reader.readtext(processed_roi, detail=0)
                    raw_runs, raw_wickets = self.extract_score(" ".join(ocr_res))
                    
                    if raw_runs is not None:
                        stable_runs, stable_wickets = self.get_stable_score(raw_runs, raw_wickets)
                        
                        if stable_runs is not None:
                            is_hl, event = self.validate_and_update(stable_runs, stable_wickets)
                            if is_hl:
                                ts = int(current_time_sec)
                                start_time = max(0, ts - 15)
                                end_time = ts + 10
                                duration = end_time - start_time
                                
                                hl_data = {
                                    "event": event,
                                    "ts": ts,
                                    "score": f"{stable_runs}/{stable_wickets}",
                                    "duration": duration,
                                    "timestamps": f"[{timedelta(seconds=start_time)}] - [{timedelta(seconds=end_time)}]"
                                }
                                
                                if event == "New Innings Detected":
                                    print(f"\n{event} | Timestamps: {hl_data['timestamps']} | Score: {hl_data['score']}")
                                self.highlights.append(hl_data)
            
            frame_count += 1
            pbar.update(1)

        cap.release()
        pbar.close()
        return self.highlights

In [ ]:
# (YOLO+OCR pipeline is invoked from the integration cell below)

# Integration

In [ ]:
import os
import re
import json
import subprocess
import torch
from dataclasses import dataclass
from typing import List, Tuple, Optional
from pydantic import BaseModel, Field
from google import genai

# =========================================================
# 1. Unified Data Models
# =========================================================

@dataclass
class VisionEvent:
    time_sec: float
    event_type: str
    score_str: str

@dataclass
class AudioPeak:
    time_sec: float
    score: float

class SemanticEvent(BaseModel):
    event_type: str = Field(description="e.g., 'Wicket', 'Six', 'Four', 'DRS Review', 'Dropped Catch'")
    approx_commentary_time: float = Field(
        description=(
            "The transcript timestamp (in seconds) where the commentator FIRST reacts to the live action. "
            "This is approximate — the actual bat crack or ball event will have occurred 2-8 seconds earlier. "
            "Use the earliest timestamp in the transcript where excitement or direct description begins. "
            "Do NOT use replay commentary timestamps."
        )
    )
    confidence: str = Field(description="'High' if OCR and commentary both confirm the event. 'Medium' if only one source supports it.")
    reason: str = Field(description="Brief explanation of how OCR and transcript corroborate this.")
    
@dataclass
class FinalClip:
    event_type: str
    start_sec: float
    end_sec: float
    reason: str

# =========================================================
# 2. LAYER 1: The Sensors
# =========================================================

def run_vision_sensor(
    video_path: str,
    yolo_model,
    ocr_reader,
    max_duration_mins: Optional[float] = None,
    persistence: int = 3,
) -> List[VisionEvent]:

    print("🎥 [SENSOR 1] Running Vision Tracker (YOLO + OCR)...")
    pipeline = CricketHighlightPipeline(
        yolo_model=yolo_model,
        ocr_reader=ocr_reader,
        video_path=video_path,
        sample_fps=1,
        max_duration_mins=max_duration_mins,
        persistence=persistence,
    )
    raw_highlights = pipeline.process_video()
    
    vision_events = []
    for hl in raw_highlights:
        if hl['event'] and hl['event'] != "New Innings Detected":
            vision_events.append(VisionEvent(
                time_sec=float(hl['ts']),
                event_type=hl['event'],
                score_str=hl['score'],
            ))
    return vision_events

def run_audio_sensor(video_path: str, work_dir: str) -> List[AudioPeak]:
    print("🔊 [SENSOR 2] Running Audio Peak Extraction...")
    os.makedirs(work_dir, exist_ok=True) 
    wav_path = os.path.join(work_dir, "temp_audio.wav")
    
    extract_audio_ffmpeg(video_path, wav_path) 
    features = compute_features_streaming(wav_path)
    score = build_composite_score(features)
    
    candidates = detect_peak_candidates(features, score)
    sustain = detect_sustained_loud_regions(features, score)
    
    peaks = []
    for c in candidates + sustain:
        peaks.append(AudioPeak(time_sec=float(c['peak_time']), score=float(c['score'])))
        
    if os.path.exists(wav_path):
        os.remove(wav_path)
        
    return peaks

# =========================================================
# 3. LAYER 2: The Brain (Targeted Gemini Verification)
# =========================================================

def get_regions_of_interest(
    vision_events: List[VisionEvent],
    transcript_segments: list,
) -> List[Tuple[float, float]]:
    """Finds time windows where something likely happened based on OCR or keywords."""
    rois = []
    
    # FIX: Tightened from 45s to 25s backward look — scoreboard lag is 10-30s, not 45s.
    # The original 45s window caused nearly the entire match to become an ROI.
    for ve in vision_events:
        rois.append([max(0.0, ve.time_sec - 25.0), ve.time_sec + 5.0])
        
    keyword_regex = re.compile(
        r"\b(review|third umpire|dropped|put down|no ball|catch it|appeal)\b",
        re.IGNORECASE,
    )
    for seg in transcript_segments:
        if keyword_regex.search(seg.text):
            rois.append([max(0.0, seg.start - 15.0), seg.end + 20.0])
            
    if not rois:
        return []
        
    rois.sort(key=lambda x: x[0])
    merged_rois = [rois[0]]
    for current in rois[1:]:
        prev = merged_rois[-1]
        if current[0] <= prev[1] + 15.0:
            prev[1] = max(prev[1], current[1])
        else:
            merged_rois.append(current)
            
    return merged_rois

def verify_events_with_gemini(
    client: genai.Client, 
    rois: List[Tuple[float, float]], 
    transcript_segments: list, 
    vision_events: List[VisionEvent],
) -> List[SemanticEvent]:
    print(f"🧠 [BRAIN] Verifying {len(rois)} targeted Regions of Interest with Gemini...")
    
    class FusionResult(BaseModel):
        events: List[SemanticEvent] = []

    all_semantic_events = []
    
    for idx, (start_t, end_t) in enumerate(rois):
        chunk_text = [
            f"[{seg.start:.1f} - {seg.end:.1f}] {seg.text}" 
            for seg in transcript_segments 
            if start_t <= seg.start <= end_t
        ]
        
        chunk_vision = [
            f"At {ve.time_sec:.1f}s: OCR Detected {ve.event_type} (Score: {ve.score_str})" 
            for ve in vision_events 
            if start_t <= ve.time_sec <= end_t
        ]
        
        vision_prompt = "\n".join(chunk_vision) if chunk_vision else "None detected by OCR in this window."
        
        # Stage 1: Flash screens the ROI cheaply
        screen_prompt = f"""Is there a genuine cricket highlight moment (wicket, boundary, DRS review, 
dropped catch) in this time window {start_t:.1f}s to {end_t:.1f}s?
Reply with only YES or NO.

[COMMENTARY]:
{chr(10).join(chunk_text)}"""

        try:
            screen_response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=screen_prompt,
                config={"temperature": 0.0},
            )
            if "NO" in screen_response.text.upper():
                continue
        except Exception as e:
            print(f"  Screening failed for ROI {start_t:.0f}-{end_t:.0f}: {e}")
            # If screening fails, proceed to verification anyway
        
        prompt = f"""You are an expert cricket video editor verifying a specific time window: {start_t:.1f}s to {end_t:.1f}s.

[VISUAL SCOREBOARD EVENTS (Delayed by 10-30s)]:
{vision_prompt}

[COMMENTARY TRANSCRIPT]:
{chr(10).join(chunk_text)}

TASK:
1. Identify any true highlight moments (Wicket, Boundary, Review, Dropped Catch, etc.).
2. Provide the EXACT timestamp the commentator FIRST starts reacting to the live action.
3. Ignore replays. If the commentator says "Let's look at that again," do not log it as a new event.
4. If nothing important happens, return an empty list."""
        
        try:
            response = client.models.generate_content(
                model="gemini-2.5-pro",
                contents=prompt,
                config={
                    "temperature": 0.1,
                    "response_mime_type": "application/json",
                    "response_json_schema": FusionResult.model_json_schema(),
                },
            )
            parsed = FusionResult.model_validate_json(response.text)
            all_semantic_events.extend(parsed.events)
            print(f"  ROI {idx+1}/{len(rois)}: {len(parsed.events)} events found")
        except Exception as e:
            print(f"  Failed to process ROI {start_t:.0f}-{end_t:.0f}: {e}")
            
    return all_semantic_events

def deduplicate_semantic_events(
    events: List[SemanticEvent],
    min_gap_seconds: float = 45.0,
) -> List[SemanticEvent]:
    """
    Removes duplicate SemanticEvents of the same type within min_gap_seconds.
    Replays typically appear 30-90s after the original — this suppresses them.
    """
    if not events:
        return []

    events_sorted = sorted(events, key=lambda e: e.approx_commentary_time)
    kept = [events_sorted[0]]

    for current in events_sorted[1:]:
        prev = kept[-1]
        same_type = current.event_type.lower() == prev.event_type.lower()
        too_close = (current.approx_commentary_time - prev.approx_commentary_time) < min_gap_seconds

        if same_type and too_close:
            print(f"🔁 Suppressed likely replay: {current.event_type} at {current.approx_commentary_time:.1f}s "
                  f"(original at {prev.approx_commentary_time:.1f}s)")
            continue

        kept.append(current)

    return kept

# =========================================================
# 4. LAYER 3: The Scalpel (Audio Anchoring)
# =========================================================

def anchor_timestamps(
    semantic_events: List[SemanticEvent],
    audio_peaks: List[AudioPeak],
    scene_detector: SceneChangeDetector = None,
) -> List[FinalClip]:
    print("✂️ [SCALPEL] Snapping verified events to Audio Peaks + Scene Boundaries...")

    PADDING_PROFILE = {
        "wicket":        {"pre": 9.0,  "post": 22.0},
        "six":           {"pre": 6.0,  "post": 14.0},
        "four":          {"pre": 5.0,  "post": 10.0},
        "drs review":    {"pre": 5.0,  "post": 75.0},
        "dropped catch": {"pre": 6.0,  "post": 18.0},
        "no ball":       {"pre": 4.0,  "post": 12.0},
        "default":       {"pre": 6.0,  "post": 16.0},
    }

    def _get_padding(event_type: str) -> dict:
        key = event_type.lower()
        for profile_key in PADDING_PROFILE:
            if profile_key in key:
                return PADDING_PROFILE[profile_key]
        return PADDING_PROFILE["default"]

    final_clips = []
    
    for event in semantic_events:
        is_high_confidence = event.confidence.lower() == "high"
        search_window_pre  = 8.0  if is_high_confidence else 14.0
        search_window_post = 2.0  if is_high_confidence else 4.0

        search_start = event.approx_commentary_time - search_window_pre
        search_end   = event.approx_commentary_time + search_window_post

        valid_peaks = [p for p in audio_peaks if search_start <= p.time_sec <= search_end]

        if valid_peaks:
            best_peak = max(valid_peaks, key=lambda x: x.score)
            anchor_time = best_peak.time_sec
        else:
            wider_peaks = [
                p for p in audio_peaks
                if (event.approx_commentary_time - 18.0) <= p.time_sec <= (event.approx_commentary_time + 5.0)
            ]
            if wider_peaks:
                best_peak = max(wider_peaks, key=lambda x: x.score)
                anchor_time = best_peak.time_sec
                print(f"⚠️  Wide-window fallback for {event.event_type} at {event.approx_commentary_time:.1f}s")
            else:
                anchor_time = event.approx_commentary_time
                print(f"⚠️  No audio peak for {event.event_type} at {event.approx_commentary_time:.1f}s — using commentary anchor.")
        
        pad = _get_padding(event.event_type)
        start_time = max(0.0, anchor_time - pad["pre"])
        end_time = anchor_time + pad["post"]

        if scene_detector and scene_detector.scene_times:
            start_time = scene_detector.snap_start(start_time, min_stable=2.0)
            end_time   = scene_detector.snap_end(end_time,     min_stable=2.0)

        final_clips.append(FinalClip(
            event_type=event.event_type,
            start_sec=round(start_time, 2),
            end_sec=round(end_time, 2),
            reason=event.reason,
        ))
        
    # Deduplicate overlapping clips
    if not final_clips:
        return []
        
    final_clips.sort(key=lambda x: x.start_sec)
    merged_clips = [final_clips[0]]
    
    for cur in final_clips[1:]:
        prev = merged_clips[-1]
        if cur.start_sec <= prev.end_sec: 
            prev.end_sec = max(prev.end_sec, cur.end_sec)
            if cur.event_type not in prev.event_type:
                prev.event_type = f"{prev.event_type} & {cur.event_type}"
        else:
            merged_clips.append(cur)
            
    return merged_clips

# =========================================================
# 5. MASTER ORCHESTRATOR
# =========================================================

def generate_unified_highlights(video_path: str, transcript_path: str, output_path: str):
    
    print("🎬 [SENSOR 0] Detecting scene boundaries (one-time pass)...")
    scene_detector = SceneChangeDetector(video_path, threshold=0.5)
    scene_detector.detect()

    # 1. Sensors
    vision_events = run_vision_sensor(video_path, trained_model, reader)
    audio_peaks = run_audio_sensor(video_path, work_dir="/kaggle/working/temp")
    transcript_segments = parse_transcript_file(transcript_path) 
    
    # 2. Brain (Targeted Fusion)
    gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
    rois = get_regions_of_interest(vision_events, transcript_segments)
    
    semantic_events = verify_events_with_gemini(
        gemini_client, rois, transcript_segments, vision_events
    )
    semantic_events = deduplicate_semantic_events(semantic_events, min_gap_seconds=45.0)

    # 3. Scalpel
    final_clips = anchor_timestamps(semantic_events, audio_peaks, scene_detector=scene_detector)
    
    # 4. Save Metadata
    metadata_out = "/kaggle/working/final_highlights_metadata.json"
    payload = [c.__dict__ for c in final_clips]
    with open(metadata_out, "w") as f:
        json.dump(payload, f, indent=2)
    print(f"📄 Saved metadata for {len(final_clips)} clips to {metadata_out}")
        
    # 5. Compilation — FIX: re-encode with audio sync flags to prevent audio delay
    print(f"🎬 [COMPILE] Extracting and merging {len(final_clips)} highlights...")
    clips_dir = "/kaggle/working/temp_clips"
    os.makedirs(clips_dir, exist_ok=True)
    clip_paths = []
    
    for idx, clip in enumerate(final_clips):
        temp_clip = os.path.join(clips_dir, f"clip_{idx:03d}.mp4")
        duration = clip.end_sec - clip.start_sec
        cmd = [
            'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
            '-ss', str(clip.start_sec),
            '-i', video_path, 
            '-t', str(duration),
            '-avoid_negative_ts', 'make_zero',          # FIX: reset PTS to 0 — no audio offset
            '-c:v', 'libx264', '-preset', 'ultrafast',  # FIX: faster than 'fast', negligible quality diff
            '-crf', '23',
            '-c:a', 'aac', '-b:a', '192k',
            '-af', 'aresample=async=1000',               # FIX: absorb residual audio drift
            '-movflags', '+faststart',
            temp_clip,
        ]
        result = subprocess.run(cmd, capture_output=True)
        
        if os.path.exists(temp_clip) and os.path.getsize(temp_clip) > 10_000:
            clip_paths.append(temp_clip)
        else:
            print(f"❌ Failed to extract clip {idx}: {clip.event_type} @ {clip.start_sec}s")
        
    concat_file = '/kaggle/working/concat.txt'
    with open(concat_file, 'w') as f:
        for cp in clip_paths:
            f.write(f"file '{os.path.abspath(cp)}'\n")
            
    merge_cmd = [
        'ffmpeg', '-y', '-hide_banner', '-loglevel', 'error',
        '-f', 'concat', '-safe', '0', '-i', concat_file, 
        '-c', 'copy',     # Safe: all clips share same encoding
        output_path,
    ]
    subprocess.run(merge_cmd, check=True)
    
    # Cleanup temp files
    for cp in clip_paths:
        if os.path.exists(cp):
            os.remove(cp)
    if os.path.exists(concat_file):
        os.remove(concat_file)
    temp_dir = "/kaggle/working/temp_clips"
    if os.path.isdir(temp_dir):
        os.rmdir(temp_dir)
        
    print(f"✅ Integrated Highlights saved successfully to {output_path}")

# =========================================================
# Execution
# =========================================================
if __name__ == "__main__":
    VIDEO_FILE = '/kaggle/input/datasets/yashkumar008/t20-full-match-dataset/ENG vs IND.mp4'
    TRANSCRIPT_FILE = '/kaggle/input/datasets/kuldipkmanvar/transcript-indvseng/transcript_with_timestamps.txt'
    OUTPUT_VIDEO = '/kaggle/working/final_integrated_highlights.mp4'
    
    generate_unified_highlights(VIDEO_FILE, TRANSCRIPT_FILE, OUTPUT_VIDEO)
